# Day 12: React UI Logic and API Integration
**Conditional rendering, lists, the Fetch API, and React talking to FastAPI**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
!pip install -q fastapi uvicorn httpx

# ===== Option B: VS Code / local =====
# pip install fastapi uvicorn httpx   (in your activated venv)
# For the separate React project: npm create vite@latest task-board -- --template react
print("Ready")

In [ ]:
# Renders a small React app inside the notebook output (works in Colab and VS Code notebooks).
import html
from IPython.display import HTML, display

def show_react(app_code, height=320):
    page = f"""<!doctype html><html><head>
<script src="https://unpkg.com/react@18/umd/react.development.js"></script>
<script src="https://unpkg.com/react-dom@18/umd/react-dom.development.js"></script>
<script src="https://unpkg.com/@babel/standalone@7/babel.min.js"></script>
<style>
  body {{ font-family: system-ui, sans-serif; margin: 16px; color: #0E3B43; }}
  .card {{ background: #E3F6F5; border-radius: 10px; padding: 12px 16px; margin: 8px 0; }}
  button {{ background: #0FA3A3; color: white; border: 0; border-radius: 6px; padding: 8px 14px; cursor: pointer; margin-right: 6px; }}
  input {{ padding: 8px; border: 1px solid #9ED9D4; border-radius: 6px; margin-right: 6px; }}
  .done {{ text-decoration: line-through; color: #6B9A9E; }}
</style></head><body><div id="root"></div>{app_code}</body></html>"""
    display(HTML(f'<iframe srcdoc="{html.escape(page, quote=True)}" style="width:100%;height:{height}px;border:1px solid #9ED9D4;border-radius:8px"></iframe>'))

print("show_react() is ready")

## 2. Conditional rendering and lists

In [ ]:
show_react("""
<script type="text/babel">
const { useState } = React;

const products = [
  { id: 1, name: "Pen", price: 20, stock: 100 },
  { id: 2, name: "Mug", price: 250, stock: 0 },
  { id: 3, name: "Lamp", price: 1200, stock: 7 },
  { id: 4, name: "Notebook", price: 80, stock: 3 },
];

function App() {
  const [loggedIn, setLoggedIn] = useState(true);
  const unread = 3;
  const sorted = [...products].sort((a, b) => a.price - b.price);
  return (
    <div>
      <p>
        {loggedIn ? "Welcome back, Mei" : "Please log in"}{" "}
        {loggedIn && unread > 0 && <b>[{unread} unread]</b>}
      </p>
      <button onClick={() => setLoggedIn(!loggedIn)}>{loggedIn ? "Log out" : "Log in"}</button>
      {sorted.map(p => (
        <div key={p.id} className="card" style={{ opacity: p.stock === 0 ? 0.5 : 1 }}>
          {p.name}: Rs {p.price} {p.stock === 0 && <em>(Out of stock)</em>}
        </div>
      ))}
    </div>
  );
}
ReactDOM.createRoot(document.getElementById("root")).render(<App />);
</script>
""", height=330)

## 3. The four states with a fake API
The fake API waits 1 second, fails 30% of the time and sometimes returns nothing. Click Retry a few times to see every state.

In [ ]:
show_react("""
<script type="text/babel">
const { useState, useEffect } = React;

function fakeApi() {
  return new Promise((resolve, reject) => {
    setTimeout(() => {
      const r = Math.random();
      if (r < 0.3) reject(new Error("Server unavailable"));
      else if (r < 0.5) resolve([]);
      else resolve([{ id: 1, title: "Read post-read" }, { id: 2, title: "Finish workbook" }]);
    }, 1000);
  });
}

function Tasks() {
  const [tasks, setTasks] = useState([]);
  const [loading, setLoading] = useState(true);
  const [error, setError] = useState(null);
  const [attempt, setAttempt] = useState(0);

  useEffect(() => {
    setLoading(true);
    setError(null);
    fakeApi()
      .then(setTasks)
      .catch(err => setError(err.message))
      .finally(() => setLoading(false));
  }, [attempt]);

  const retry = <button onClick={() => setAttempt(a => a + 1)}>Retry</button>;
  if (loading) return <p>Loading tasks...</p>;
  if (error) return <div className="card">Could not load tasks: {error} {retry}</div>;
  if (tasks.length === 0) return <div className="card">No tasks yet. {retry}</div>;
  return (
    <div>
      {tasks.map(t => <div key={t.id} className="card">{t.title}</div>)}
      {retry}
    </div>
  );
}
ReactDOM.createRoot(document.getElementById("root")).render(<Tasks />);
</script>
""", height=220)

## 4. A full stack Task Board
FastAPI serves both the API (`/api/tasks`) and the React page (`/`). Because they share one origin, no CORS is needed here; CORS is still added for the separate Vite setup.

In [ ]:
%%writefile index.html
<!doctype html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Task Board</title>
  <script src="https://unpkg.com/react@18/umd/react.development.js"></script>
  <script src="https://unpkg.com/react-dom@18/umd/react-dom.development.js"></script>
  <script src="https://unpkg.com/@babel/standalone@7/babel.min.js"></script>
  <style>
    body { font-family: system-ui, sans-serif; margin: 24px; color: #0E3B43; max-width: 640px; }
    .card { background: #E3F6F5; border-radius: 10px; padding: 10px 14px; margin: 8px 0; display: flex; gap: 8px; align-items: center; }
    .done { text-decoration: line-through; color: #6B9A9E; }
    button { background: #0FA3A3; color: #fff; border: 0; border-radius: 6px; padding: 6px 12px; cursor: pointer; }
    button.del { background: #FF8A3D; margin-left: auto; }
    input { padding: 8px; border: 1px solid #9ED9D4; border-radius: 6px; }
  </style>
</head>
<body>
<div id="root"></div>
<script type="text/babel">
const { useState, useEffect } = React;
const API = "";   // same origin. With Vite on port 5173, set this to your API address on port 8000

function TaskBoard() {
  const [tasks, setTasks] = useState([]);
  const [loading, setLoading] = useState(true);
  const [error, setError] = useState(null);
  const [title, setTitle] = useState("");
  const [saving, setSaving] = useState(false);
  const [query, setQuery] = useState("");

  useEffect(() => {
    fetch(API + "/api/tasks")
      .then(r => { if (!r.ok) throw new Error(r.status); return r.json(); })
      .then(setTasks)
      .catch(e => setError(e.message))
      .finally(() => setLoading(false));
  }, []);

  async function add(e) {
    e.preventDefault();
    if (!title.trim()) return;
    setSaving(true);
    const r = await fetch(API + "/api/tasks", {
      method: "POST", headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ title }),
    });
    if (r.ok) { const created = await r.json(); setTasks(prev => [...prev, created]); setTitle(""); }
    setSaving(false);
  }

  async function toggle(t) {
    const r = await fetch(`${API}/api/tasks/${t.id}`, {
      method: "PATCH", headers: { "Content-Type": "application/json" },
      body: JSON.stringify({ done: !t.done }),
    });
    if (r.ok) { const updated = await r.json(); setTasks(prev => prev.map(x => x.id === t.id ? updated : x)); }
  }

  async function remove(id) {
    const r = await fetch(`${API}/api/tasks/${id}`, { method: "DELETE" });
    if (r.ok) setTasks(prev => prev.filter(x => x.id !== id));
  }

  if (loading) return <p>Loading tasks...</p>;
  if (error) return <p>Could not load tasks: {error}</p>;
  const visible = tasks.filter(t => t.title.toLowerCase().includes(query.toLowerCase()));

  return (
    <div>
      <h1>Task Board</h1>
      <form onSubmit={add}>
        <input value={title} onChange={e => setTitle(e.target.value)} placeholder="New task" />{" "}
        <button disabled={saving}>{saving ? "Saving..." : "Add"}</button>
      </form>
      <p><input value={query} onChange={e => setQuery(e.target.value)} placeholder="Search" /></p>
      {visible.length === 0 && <p>No tasks match.</p>}
      {visible.map(t => (
        <div key={t.id} className="card">
          <input type="checkbox" checked={t.done} onChange={() => toggle(t)} />
          <span className={t.done ? "done" : ""}>{t.title}</span>
          <button className="del" onClick={() => remove(t.id)}>Delete</button>
        </div>
      ))}
    </div>
  );
}
ReactDOM.createRoot(document.getElementById("root")).render(<TaskBoard />);
</script>
</body>
</html>

In [ ]:
%%writefile fullstack.py
from pathlib import Path
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field

app = FastAPI(title="Task Board API")
app.add_middleware(CORSMiddleware, allow_origins=["http://localhost:5173", "http://127.0.0.1:5173"],
                   allow_methods=["*"], allow_headers=["*"])

class TaskIn(BaseModel):
    title: str = Field(min_length=1, max_length=100)

class TaskPatch(BaseModel):
    title: str | None = None
    done: bool | None = None

class Task(BaseModel):
    id: int
    title: str
    done: bool = False

tasks: dict[int, Task] = {1: Task(id=1, title="Read the Day 12 post-read"), 2: Task(id=2, title="Try the search box")}

@app.get("/", response_class=HTMLResponse)
def page():
    return Path("index.html").read_text()

@app.get("/api/tasks", response_model=list[Task])
def list_tasks():
    return list(tasks.values())

@app.post("/api/tasks", response_model=Task, status_code=201)
def create(data: TaskIn):
    new_id = max(tasks, default=0) + 1
    tasks[new_id] = Task(id=new_id, title=data.title)
    return tasks[new_id]

@app.patch("/api/tasks/{tid}", response_model=Task)
def patch(tid: int, data: TaskPatch):
    if tid not in tasks:
        raise HTTPException(404, "Task not found")
    tasks[tid] = tasks[tid].model_copy(update=data.model_dump(exclude_unset=True))
    return tasks[tid]

@app.delete("/api/tasks/{tid}", status_code=204)
def delete(tid: int):
    if tasks.pop(tid, None) is None:
        raise HTTPException(404, "Task not found")

### Test the API first

In [ ]:
import importlib, fullstack
importlib.reload(fullstack)
from fastapi.testclient import TestClient
c = TestClient(fullstack.app)

print(c.get("/api/tasks").json())
print(c.post("/api/tasks", json={"title": "Demo task"}).status_code)
print(c.patch("/api/tasks/1", json={"done": True}).json())
print("Empty title:", c.post("/api/tasks", json={"title": ""}).status_code)
print("Page served:", "text/babel" in c.get("/").text)
r = c.options("/api/tasks", headers={"Origin": "http://localhost:5173", "Access-Control-Request-Method": "POST"})
print("CORS allowed origin:", r.headers.get("access-control-allow-origin"))

### Run the full stack app
**Colab:** the cell below starts the server and opens the app inside the notebook.

**VS Code:** run `uvicorn fullstack:app --reload` in the terminal and open http://127.0.0.1:8000 in your browser.

In [ ]:
# ===== Option A: Google Colab (active) =====
import subprocess, time
server = subprocess.Popen(["uvicorn", "fullstack:app", "--port", "8000"])
time.sleep(3)
from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=500)

# ===== Option B: VS Code =====
# Terminal: uvicorn fullstack:app --reload
# Browser:  http://127.0.0.1:8000

### TODO 1: Add a "Clear completed" button that deletes every done task (call DELETE for each, or add a new API endpoint).

In [ ]:
# TODO: write your code here

## 5. The separate Vite version (VS Code)
In a real project, React lives in its own Vite app. Copy the `TaskBoard` component from `index.html` into `src/App.jsx`, add `import { useState, useEffect } from "react";`, set `const API = "http://127.0.0.1:8000";`, and export it with `export default TaskBoard;`. Run `npm run dev` and `uvicorn fullstack:app --reload` side by side. CORS is already configured for port 5173.

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)